In [1]:
import pandas as pd
import splink.comparison_library as cl
from splink import Linker, SettingsCreator, block_on, DuckDBAPI

# Load data
data = pd.read_csv("csv_outputs/clean_data_owner_president_ceo_founder_2026-01-14.csv", dtype="string")
# df = data[data["company_state"] == "Oklahoma"].copy()
df = data.copy()
df["company_employee_size_actual"] = pd.to_numeric(df["company_employee_size_actual"], errors="coerce")
print(f"Total rows: {len(df):,}")

# Add unique ID required by splink - already crreated with record_id

# Clean key columns for better matching
df["first_name_clean"] = df["first_name"].str.replace(r'[^a-zA-Z]', '', regex=True).str.strip().str.lower()
df["last_name_clean"] = df["last_name"].str.replace(r'[^a-zA-Z]', '', regex=True).str.strip().str.lower()
df["forename_surname_concat_col_name"] = df["first_name_clean"] + " " + df["last_name_clean"]
df["company_name_clean"] = df["company_name"].str.replace(r'[^a-zA-Z0-9]', '', regex=True).str.strip().str.lower()
df["email_clean"] = df["email"].str.replace(" ", "").str.lower().str.strip()
df['linkedin_clean'] = df['linkedin'].str.split("/in/").str[-1].str.lower()

Total rows: 1,621,049


In [2]:
# Define splink settings
# Blocking rules reduce the number of comparisons dramatically
# These rules determine which record pairs to compare

settings = SettingsCreator(
    link_type="dedupe_only",
    unique_id_column_name="record_id",
    
    # Blocking rules - records must match on at least one of these to be compared 
    blocking_rules_to_generate_predictions=[
        block_on("first_name_clean", "last_name_clean"),
        block_on("first_name_clean", "company_name_clean"),
        block_on("last_name_clean", "company_name_clean", "email_clean"),
        block_on("linkedin_clean", "company_name_clean"),
    ],
    comparisons=[
        cl.ForenameSurnameComparison("first_name_clean", "last_name_clean", forename_surname_concat_col_name="forename_surname_concat_col_name"),
        cl.ExactMatch("email_clean"),
        cl.ExactMatch("linkedin_clean"),
        cl.NameComparison("company_name_clean"),
        cl.ExactMatch("company_zipcode"),
    ],
    
    # Retain these columns in results
    retain_intermediate_calculation_columns=True,
)
# Initialize the linker with DuckDB backend (fast, in-memory)
linker = Linker(df, settings, db_api=DuckDBAPI())


In [3]:

# Estimate probability two random records match (u probability)
# This uses random sampling, so it's fast
linker.training.estimate_u_using_random_sampling(max_pairs=int(1e8))

# Estimate m probabilities using Expectation Maximization
# This learns how likely matching records are to agree on each field

linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("first_name_clean", "last_name_clean")
)

linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("first_name_clean", "company_name_clean")
)

linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("email_clean") 
)

linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("linkedin_clean")
)

linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("company_zipcode", "last_name_clean")  
)

----- Estimating u probabilities using random sampling -----

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - first_name_clean_last_name_clean (no m values are trained).
    - email_clean (no m values are trained).
    - linkedin_clean (no m values are trained).
    - company_name_clean (no m values are trained).
    - company_zipcode (no m values are trained).

----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
(l."first_name_clean" = r."first_name_clean") AND (l."last_name_clean" = r."last_name_clean")

Parameter estimates will be made for the following comparison(s):
    - email_clean
    - linkedin_clean
    - company_name_clean
    - company_zipcode

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - first_name_clean_last_name_clean

Iteration 1: Largest change in params was 0.296 in probability_two_

<EMTrainingSession, blocking on (l."company_zipcode" = r."company_zipcode") AND (l."last_name_clean" = r."last_name_clean"), deactivating comparisons first_name_clean_last_name_clean, company_zipcode>

In [4]:

linker.visualisations.match_weights_chart()


c:\Projects\Freelanxur\upwork_42168194-data_merge\.venv-upwork_42168194-data_merge\Lib\site-packages\altair\vegalite\v6\api.py:4124: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  return _tp.from_dict(dct, validate=validate)


alt.VConcatChart(...)

In [5]:
# Predict matches - this is where the actual deduplication happens
# threshold controls minimum match probability (0.0 to 1.0)
# Lower = more matches (but more false positives)
# Higher = fewer matches (but may miss some true duplicates)

print("Running predictions... (this may take a few minutes for 1.4M rows)")
predictions = linker.inference.predict(threshold_match_probability=0.95) # this is just a filter to store less data
pairwise_predictions = predictions.as_pandas_dataframe()
print("Predictions complete!")


Running predictions... (this may take a few minutes for 1.4M rows)


Blocking time: 10.39 seconds
Predict time: 13.08 seconds


Predictions complete!


In [6]:

# Cluster the predictions into groups of duplicates
# Each cluster represents a unique entity
clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    predictions, 
    threshold_match_probability=0.95 # this is probably that connects records
)

df_with_clusters = clusters.as_pandas_dataframe().astype("string")
print(f"Number of clusters: {df_with_clusters['cluster_id'].nunique():,}")
print(f"Records in clusters: {len(df_with_clusters):,}")
df_with_clusters.to_csv("csv_outputs/df_with_clusters.csv", index=False)
df_with_clusters

Completed iteration 1, num edges remaining to process: 18328
Completed iteration 2, num edges remaining to process: 564
Completed iteration 3, num edges remaining to process: 82
Completed iteration 4, num edges remaining to process: 14
Completed iteration 5, num edges remaining to process: 4
Completed iteration 6, num edges remaining to process: 0


Number of clusters: 1,046,818
Records in clusters: 1,621,049


,cluster_id,source_file,record_id,first_name,last_name,job_title,email,mobile_phone,company_linkedin,facebook,...,primary_2022,general_2020,primary_2020,home_zipfour,first_name_clean,last_name_clean,forename_surname_concat_col_name,company_name_clean,email_clean,linkedin_clean
0,1218115,[STATE] raw_data_states\slack\arkansas 501 8.xlsx,1218115,Jillian,Carlan,President,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,jillian,carlan,jillian carlan,jilliancarlancounselingpllc,<NA>,<NA>
1,1063467,[STATE] raw_data_states\2025-12-20\pennsylvani...,1063467,Jillian,Tuskweth,Owner,<NA>,<NA>,<NA>,https://www.facebook.com/pages/jillian-grace-s...,...,<NA>,<NA>,<NA>,<NA>,jillian,tuskweth,jillian tuskweth,jilliangracesalon,<NA>,<NA>
2,1250264,[STATE] raw_data_states\slack\florida 561 13.xlsx,1250264,Jillian,Frieder,Owner,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,jillian,frieder,jillian frieder,jillianhfriedermddrmtlgy,<NA>,<NA>
3,1016774,[STATE] raw_data_states\2025-12-20\new mexico....,1016774,Wayne,Brry,Owner,<NA>,<NA>,<NA>,https://facebook.com/jillianhomesnm,...,<NA>,<NA>,<NA>,<NA>,wayne,brry,wayne brry,jillianhomes,<NA>,<NA>
4,1087971,[STATE] raw_data_states\2025-12-20\south carol...,1087971,Dylan,Huff,Owner,<NA>,<NA>,<NA>,http://www.facebook.com/jillians.columbia,...,<NA>,<NA>,<NA>,<NA>,dylan,huff,dylan huff,jillians,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1621044,541424,Zoominfo scarping.xlsx - Texas.csv,541424,Jim,<NA>,Owner,jcloud@slimdril.com,19157279709,<NA>,https://www.facebook.com/profile.php?id=100028...,...,<NA>,<NA>,<NA>,<NA>,jim,<NA>,<NA>,slimdril,jcloud@slimdril.com,jim-cloud-92a39a34
1621045,1811737,Zoominfo scraping list 2 - Florida.csv,645646,Mcfarlane,<NA>,Chief Executive Officer,rj@smallax.com,'+1 408-627-3430,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,mcfarlane,<NA>,<NA>,smallaxcreative,rj@smallax.com,r-j-mcfarlane-89563256
1621046,1811737,Combined Master File 2_1,1811737,Mcfarlane,<NA>,Chief Executive Officer,rj@smallax.com,+1 408-627-3430,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,mcfarlane,<NA>,<NA>,smallaxcreative,rj@smallax.com,r-j-mcfarlane-89563256
1621047,118657,allin1match file no 2 - Sheet1 FIXED.csv,118657,<NA>,<NA>,Owner,gary@northpointcm.com,<NA>,<NA>,https://www.facebook.com/northpointcm/,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,northpointconstructionmanagement,gary@northpointcm.com,gary-thomas-661ba07


In [7]:
# Define priority for source files 1 = best, 2 = middle, 3 = worst
df_with_clusters["source_file_sort_priority"] = 2  # default
df_with_clusters.loc[df_with_clusters["source_file"].str.contains("[STATE]", regex=False, na=False), "source_file_sort_priority"] = 1
df_with_clusters.loc[df_with_clusters["source_file"].isin(["Combined Master File 2_1", "www.csv"]), "source_file_sort_priority"] = 3

df_with_clusters = df_with_clusters.sort_values(by="source_file_sort_priority", ascending=True)
df_with_clusters = df_with_clusters.drop(columns=["source_file_sort_priority"] + [x for x in df_with_clusters.columns.tolist() if x.endswith("_clean")])

# Define aggregation rules based on column name patterns
# Format: (condition_function, aggregation_function)
# The condition_function takes a column name and returns True if the rule applies
# The aggregation_function is the pandas agg function to use

def agg_join(series):
    return series.tolist()

def agg_longest(series):
    """Return the longest non-null string value"""
    non_null = series.dropna()
    if len(non_null) == 0:
        return None
    return max(non_null, key=len)

def agg_phone_numbers(series):
    non_null = series.dropna()
    if len(non_null) == 0:
        return None
    
    # Priority 1: Has ( or )
    mask = non_null.str.contains(r'[()]', regex=True, na=False)
    if mask.any():
        return non_null[mask].iloc[0]
    
    # Priority 2: Has +
    mask = non_null.str.contains(r'\+', regex=True, na=False)
    if mask.any():
        return non_null[mask].iloc[0]
    
    # Priority 3: Longest value
    if len(non_null) > 0:
        longest_val = max(non_null, key=len)
        return longest_val
    
    return non_null.iloc[0]

def agg_mode(series):
    """Return the most frequently occurring non-null value"""
    non_null = series.dropna()
    if len(non_null) == 0:
        return None
    mode_result = non_null.mode()
    return mode_result.iloc[0] if len(mode_result) > 0 else non_null.iloc[0]

# ============================================================
# CUSTOMIZE YOUR AGGREGATION RULES HERE
# ============================================================
# Each rule is a tuple: (condition, aggregation_name/function)
# Conditions are checked in order - first match wins
# Available built-in agg names: 'first', 'last', 'min', 'max', 'sum', 'mean', 'count'
# Or use custom functions defined above

aggregation_rules = [
    # Example: Join all unique_ids together
    (lambda col: col == "source_file", agg_join),
    (lambda col: col == 'record_id', agg_join),
    (lambda col: 'work_phone' in col.lower(), agg_phone_numbers),
    (lambda col: 'mobile_phone' in col.lower(), agg_phone_numbers),
    (lambda col: 'landline_phone' in col.lower(), agg_phone_numbers),
    (lambda col: 'description' in col.lower(), agg_longest),
    # (lambda col: 'address' in col.lower(), agg_longest),
    (lambda col: 'actual' in col.lower(), "max"),
    (lambda col: 'volume' in col.lower(), "max"),
    (lambda col: 'revenue' in col.lower(), "max"),
    (lambda col: col == 'age', "max"),
]

# Default aggregation for columns that don't match any rule
default_agg = 'first'

# Build the aggregation dictionary for each column
def get_agg_function(col_name):
    """Return the appropriate aggregation function for a column"""
    for condition, agg_func in aggregation_rules:
        if condition(col_name):
            return agg_func
    return default_agg

# Get columns to aggregate (exclude group columns)
agg_cols = [col for col in df_with_clusters.columns if col != "cluster_id"]

# Build aggregation dict
agg_dict = {col: get_agg_function(col) for col in agg_cols}

agg_dict_readable = {}
for col, func in agg_dict.items():
    func_name = func.__name__ if hasattr(func, '__name__') else str(func)
    agg_dict_readable[col] = func_name

# Perform the deduplication with groupby and agg
print(f"Original rows: {len(df_with_clusters)}")

# df_with_clusters = df_with_clusters.astype("string")
df_deduped = df_with_clusters.groupby("cluster_id", dropna=False).agg(agg_dict).reset_index()

print(f"Deduplicated rows: {len(df_deduped)}")
print(f"Duplicates removed: {len(df_with_clusters) - len(df_deduped)}")

# df = df.sort_values(by=cols_to_sort_drop).drop(columns=cols_to_sort_drop)
# df_deduped = df_deduped.sort_values(by=cols_to_sort_drop).drop(columns=cols_to_sort_drop)


Original rows: 1621049
Deduplicated rows: 1046818
Duplicates removed: 574231


In [8]:
df_with_clusters

,cluster_id,source_file,record_id,first_name,last_name,job_title,email,mobile_phone,company_linkedin,facebook,...,voting_performance_minor_election,primary_n_of_4,general_2024,primary_2024,general_2022,primary_2022,general_2020,primary_2020,home_zipfour,forename_surname_concat_col_name
657537,1194751,[STATE] raw_data_states\Sasha Files\Wisconsin\...,1194751,Kelly,Brown,Chief Executive Officer,<NA>,<NA>,<NA>,https://www.facebook.com/depositmanagement,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,kelly brown
657532,1180051,[STATE] raw_data_states\Sasha Files\Virginia\v...,1180051,Jerry,Bates,Owner,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,jerry bates
657589,1135045,[STATE] raw_data_states\2025-12-20\washingtond...,1135045,Tim,Phillips,President,<NA>,<NA>,<NA>,http://www.facebook.com/afpwa,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,tim phillips
657581,1382025,[STATE] raw_data_states\slack\Michigan 989 .xlsx,1382025,Abdul,Khan,Owner,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,abdul khan
657580,1135047,[STATE] raw_data_states\2025-12-20\washingtond...,1135047,Robert,Lynch,President,<NA>,<NA>,http://www.linkedin.com/company/americans-for-...,https://www.facebook.com/americans4arts,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,robert lynch
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1621048,118657,www.csv,1717331,<NA>,<NA>,Owner,gary@northpointcm.com,<NA>,http://www.linkedin.com/company/northpointcm,https://www.facebook.com/northpointcm/,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
257372,1813601,Combined Master File 2_1,1813601,Salvatore,Rosenblatt,President,sal@smo.marketing,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,'4034,salvatore rosenblatt
196024,1727166,Combined Master File 2_1,1727166,Pete,Kaplan,President,pkaplan@pkfinancialgroup.com,+1 216-402-4631,<NA>,<NA>,...,0%,0,Y,<NA>,Y,<NA>,Y,<NA>,<NA>,pete kaplan
87323,118432,www.csv,1595781,Justin,Hunter,President,justin@hollygc.com,<NA>,http://www.linkedin.com/company/holly-construc...,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,justin hunter


In [9]:
df_deduped

,cluster_id,source_file,record_id,first_name,last_name,job_title,email,mobile_phone,company_linkedin,facebook,...,voting_performance_minor_election,primary_n_of_4,general_2024,primary_2024,general_2022,primary_2022,general_2020,primary_2020,home_zipfour,forename_surname_concat_col_name
0,1,[3rd Muhammad.csv],[1],Terrel,Davis,Chief Executive Officers,terrel.davis@ttimemanagementllc.com,<NA>,<NA>,https://facebook.com/anexinet,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,terrel davis
1,10,[3rd Muhammad.csv],[10],Jeremy,Smithson,Chief Executive Officer,jeremy@stitchyfish.com,<NA>,<NA>,https://www.facebook.com/shopstitchyfish/,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,jeremy smithson
2,100,[[STATE] raw_data_states\Sasha Files\Alabama\a...,"[1146958, 100, 572866]",Kevin,Weber,President,kevin@oxfoundations.com,+1 205-966-3161,<NA>,http://www.facebook.com/a-1-foundation-solutio...,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,kevin weber
3,1000,"[3rd Muhammad.csv, allin1match file no 2 - Sh...","[1000, 52837, 331183, 1585870, 1857698]",Bryce,Wood,President,bryce@tcboiler.com,+1 251-751-6260,http://www.linkedin.com/company/tc-boiler-piping,https://www.facebook.com/p/tc-boiler-piping-10...,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,bryce wood
4,10000,[3rd Muhammad.csv],[10000],Shani,Dowell,Founder & Chief Executive Officer,shani@possipit.com,+1 713-659-9549,<NA>,http://facebook.com/possipit,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,shani dowell
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1046813,999991,[[STATE] raw_data_states\2025-12-20\montana.xlsx],[999991],K D,Dickinson,Owner,<NA>,<NA>,<NA>,https://www.facebook.com/porticomissoula,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,kd dickinson
1046814,999992,[[STATE] raw_data_states\2025-12-20\montana.xlsx],[999992],Jack,Tafolla,President,<NA>,<NA>,<NA>,https://www.facebook.com/postframebuilding,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,jack tafolla
1046815,999994,[[STATE] raw_data_states\2025-12-20\montana.xl...,"[999995, 999994]",Drew,Stobie,Owner,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,drew stobie
1046816,999996,[[STATE] raw_data_states\2025-12-20\montana.xlsx],[999996],Mike,Rossell,President,<NA>,<NA>,<NA>,http://www.facebook.com/pottersfieldministries,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,mike rossell


In [10]:
df_deduped.to_csv("csv_outputs/deduped_data_owner_president_ceo_founder.csv", index=False)
# df_deduped.head(10000).to_csv("csv_outputs/deduped_data_owner_president_ceo_founder_sample.csv", index=False)
# df.to_csv("csv_outputs/oklahoma_before_dedupe.csv", index=False)
# df_deduped.to_csv("csv_outputs/oklahoma_after_dedupe.csv", index=False)
